# Gene Insert Metrics -- Coverage and Collinearity (VIF)

Per-gene insert coverage counts (total / sense / antisense) plus a Variance Inflation Factor
(VIF) that flags genes whose individual fitness-effect estimate is unreliable because they're
almost always co-covered by the same fragments as their neighbors.

Ported from PPAT-018/00_Gene_Insert_Prep.ipynb's `LB_Salt_Gene_Metrics` section. The original
notebook builds its input from raw per-plate S3 parquet files (T1 filter, replicate-count filter,
sense/antisense derivation); `data/selection_experiment_insert_data.parquet` is already that same
prepared, LB_4_salt-only table, so this notebook skips straight to the metrics computation.


## Configuration

In [ ]:
from pathlib import Path

# --- data directory (populate yourself -- see README's Data section) ---
DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- paths ---
INPUT_PATH        = DATA_DIR / "selection_experiment_insert_data.parquet"
GENE_COORDS_PATH  = DATA_DIR / "gene_fitness_results_with_annotations.parquet"
OUTPUT_PATH       = RESULTS_DIR / "gene_insert_metrics.parquet"

# --- parameters ---
ENVIRONMENT      = "LB_4_salt"
COORD_WINDOW_BP  = 5000


## Load data

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from tqdm import tqdm

fitness_data = pd.read_parquet(INPUT_PATH)
fitness_data_filt = fitness_data[fitness_data["environment"] == ENVIRONMENT].rename(
    columns={"bc_sequence": "insert_id"}
)
print(f"{len(fitness_data_filt):,} rows for {ENVIRONMENT}")

### Gene coordinate lookup -- derived from the public gene-level fitness table, which carries
### one row per gene with gene_id/gene_chrom/gene_start/gene_end (same columns the original
### notebook read from a private S3 gene-annotation parquet)
gene_coords_df = pd.read_parquet(GENE_COORDS_PATH)
gene_coords: dict[str, tuple[str, int, int]] = {
    row.gene_id: (row.gene_chrom, int(row.gene_start), int(row.gene_end))
    for row in gene_coords_df.itertuples(index=False)
}


## Gene coverage metrics

In [ ]:
### Gene-insert relationships (for building indicator matrices)
gene_matrix_long = (
    fitness_data_filt[fitness_data_filt["insert_type"] == "aligned"][["insert_id", "gene_ids_fully_covered"]]
    .explode("gene_ids_fully_covered")
    .drop_duplicates()
    .dropna()
)

gene_sense_long = (
    fitness_data_filt[fitness_data_filt["insert_type"] == "aligned"][["insert_id", "sense_gene_ids"]]
    .explode("sense_gene_ids")
    .drop_duplicates()
    .dropna()
)

gene_antisense_long = (
    fitness_data_filt[fitness_data_filt["insert_type"] == "aligned"][["insert_id", "antisense_gene_ids"]]
    .explode("antisense_gene_ids")
    .drop_duplicates()
    .dropna()
)

gene_insert_count    = gene_matrix_long.groupby("gene_ids_fully_covered")["insert_id"].count().to_dict()
gene_sense_count     = gene_sense_long.groupby("sense_gene_ids")["insert_id"].count().to_dict()
gene_antisense_count = gene_antisense_long.groupby("antisense_gene_ids")["insert_id"].count().to_dict()

gene_metrics = pd.DataFrame.from_dict(
    {"n_inserts": gene_insert_count, "n_sense_inserts": gene_sense_count, "n_antisense_inserts": gene_antisense_count},
    orient="columns",
).reset_index().rename(columns={"index": "gene_id"})

gene_metrics.fillna(0, inplace=True)
gene_metrics["environment"] = ENVIRONMENT

assert all(gene_metrics["n_inserts"] == gene_metrics["n_sense_inserts"] + gene_metrics["n_antisense_inserts"]), \
    "n_inserts != n_sense_inserts + n_antisense_inserts"

print(f"{len(gene_metrics):,} genes with \u22651 fully-covering insert")
gene_metrics.head()


## Variance Inflation Factor (VIF)

For each gene, build a binary fragment x gene indicator matrix from all aligned inserts within
`COORD_WINDOW_BP` of the gene, then compute VIF = 1 / (1 - R²) for that gene's column against its
neighbors. VIF = 1 means no collinearity; a high VIF means the gene is almost always co-covered by
the same fragments as its neighbors, making its individual fitness-effect estimate unreliable.

In [ ]:
def get_gene_VIF(gene_ID, gene_coords, insert_coords, insert_to_genes):
    chrom, gene_start, gene_stop = gene_coords[gene_ID]
    window_start = gene_start - COORD_WINDOW_BP
    window_end   = gene_stop  + COORD_WINDOW_BP

    # --- Select aligned inserts with at least one endpoint inside the padded window
    #     (or that fully span it) ---
    selected_inserts = [
        ins_id for ins_id, (c, s, e) in insert_coords.items()
        if c == chrom and (window_start <= s <= window_end or window_start <= e <= window_end
                            or (s <= window_start and e >= window_end))
    ]

    # --- Build binary indicator matrix for genes fully covered by selected inserts ---
    genes_in_window = sorted(frozenset().union(*(insert_to_genes.get(i, frozenset()) for i in selected_inserts)))
    gene_idx = {g: j for j, g in enumerate(genes_in_window)}

    indicator = np.zeros((len(selected_inserts), len(genes_in_window)), dtype=np.int8)
    genes_set = set(genes_in_window)
    for i, ins_id in enumerate(selected_inserts):
        for gene in (insert_to_genes.get(ins_id, frozenset()) & genes_set):
            indicator[i, gene_idx[gene]] = 1

    gene_insert_df = pd.DataFrame(indicator, index=selected_inserts, columns=genes_in_window)

    try:
        VIF_df = gene_insert_df.copy()
        VIF_df["intercept"] = 1
        gene_VIF = variance_inflation_factor(VIF_df.values, VIF_df.columns.get_loc(gene_ID))
    except Exception:
        print(f"Error calculating VIF for gene {gene_ID}")
        gene_VIF = np.nan

    return gene_VIF


In [ ]:
insert_to_genes = (
    gene_matrix_long
    .groupby("insert_id")["gene_ids_fully_covered"]
    .apply(frozenset).to_dict()
)

### Per-insert coordinate lookup (aligned inserts only)
aligned_coords = fitness_data_filt[fitness_data_filt["insert_type"] == "aligned"][
    ["insert_id", "reference_name", "reference_start", "reference_end"]
].drop_duplicates(subset="insert_id")

insert_coords: dict[str, tuple[str, int, int]] = {
    row.insert_id: (row.reference_name, int(row.reference_start), int(row.reference_end))
    for row in aligned_coords.itertuples(index=False)
}

gene_VIF = {}
for gene in tqdm(gene_metrics["gene_id"]):
    gene_VIF[gene] = get_gene_VIF(gene, gene_coords, insert_coords, insert_to_genes)

gene_metrics["VIF"] = gene_metrics["gene_id"].map(gene_VIF)

print(f"{gene_metrics['VIF'].isna().sum()} genes with NaN VIF")
gene_metrics.head()


In [ ]:
gene_metrics.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved {len(gene_metrics):,} rows to {OUTPUT_PATH}")
